In [1]:
# import os
# os.chdir("/content")

In [2]:
!ls

drive  sample_data  structural_repair.py


In [3]:
# !mkdir -p "/content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b_second/repair_applied"
# !mkdir -p "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p_test/repair_applied"
!mkdir -p "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/repair_applied"

In [4]:
!python structural_repair.py \
    --input "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/predictions.jsonl" \
    --output "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/repair_applied/predictions_repaired.jsonl" \
    --report "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/repair_applied/repair_report.json" \
    --broken "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/repair_applied/still_broken.json" \
    --manifest "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/repair_applied/change_manifest.json"

Loading predictions from: /content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/predictions.jsonl

=== REPAIR STATUS SUMMARY ===
  valid_raw           :    459  (15.28%)
  fixed_valid         :   2525  (84.05%)
  invalid_schema      :     20  ( 0.67%)

=== STRUCTURAL FIXES APPLIED (by total occurrences) ===
  box_flat_list_reshaped                  :   2967 occurrence(s) across  1884 record(s)
  json_truncation_repaired                :   1512 occurrence(s) across  1512 record(s)
  json_syntax_parentheses_boxes           :   1226 occurrence(s) across  1219 record(s)
  box_string_coordinates_parsed           :    528 occurrence(s) across   242 record(s)
  box_dropped_unreconstructable           :    388 occurrence(s) across   114 record(s)
  box_string_missing_closing_quote_repaired:    232 occurrence(s) across   143 record(s)
  box_dict_corners_converted              :    116 occurrence(s) across    86 record(s)
  box_string_multi_box_reshaped           :

In [5]:
import json

path = "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/repair_applied/predictions_repaired.jsonl"

counts = {}
with open(path, "r") as f:
    for line in f:
        if line.strip():
            r = json.loads(line)
            status = r.get("repair_status", "unknown")
            counts[status] = counts.get(status, 0) + 1

print(f"Total records: {sum(counts.values())}")
for status, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {status}: {count}")

Total records: 3004
  fixed_valid: 2525
  valid_raw: 459
  invalid_schema: 20


In [6]:
import json

path = "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/repair_applied/predictions_repaired.json"

counts = {}
with open(path, "r") as f:
    # Load the entire standard JSON file into memory
    data = json.load(f)

    # Iterate through the records (assuming data is a list of dictionaries)
    for r in data:
        status = r.get("repair_status", "unknown")
        counts[status] = counts.get(status, 0) + 1

print(f"Total records: {sum(counts.values())}")
for status, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {status}: {count}")

Total records: 3004
  fixed_valid: 2525
  valid_raw: 459
  invalid_schema: 20


In [ ]:
import os
import json
import re
from typing import Any, Dict, Optional, List
from pydantic import BaseModel, Field, conlist
from collections import Counter


PREDICTIONS_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/repair_applied/predictions_repaired.jsonl"
assert os.path.exists(PREDICTIONS_PATH), f"File not found: {PREDICTIONS_PATH}"
print(f"Found predictions file: {PREDICTIONS_PATH}")
print(f"File size: {os.path.getsize(PREDICTIONS_PATH) / 1e6:.2f} MB")

Found predictions file: /content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p_test/repair_applied/predictions_repaired.jsonl
File size: 8.18 MB


In [ ]:
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"[WARNING] Skipping unreadable line {line_num} in the .jsonl file itself: {e}")
    return records

records = load_jsonl(PREDICTIONS_PATH)
print(f"Loaded {len(records)} total records.")
print("\nSample record keys:", list(records[0].keys()) if records else "NO RECORDS")

Loaded 3004 total records.

Sample record keys: ['image_id', 'raw_output', 'sample', 'latency_seconds', 'repair_status']


In [ ]:
def strip_fences(text: str) -> str:
    """Strips markdown code fences (```json ... ```) from a string.
    Uses regex to extract content between fences, ignoring any pre/post text."""
    match = re.search(r"```(?:json)?(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return text.strip()


def parse_model_output(raw_str: str) -> Optional[Dict[str, Any]]:
    """Parses a raw string from the VLM into a dict. Returns None on failure."""
    if not raw_str or not raw_str.strip():
        return None
    text = strip_fences(raw_str)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


# Run Gate 1 over every record
parsed_results = []
for r in records:
    raw = r.get("raw_output", "")
    parsed = parse_model_output(raw)
    parsed_results.append(parsed)

n_json_valid = sum(1 for p in parsed_results if p is not None)
n_json_invalid = len(records) - n_json_valid

print("=" * 60)
print("GATE 1 — JSON PARSING")
print("=" * 60)
print(f"Total records:        {len(records)}")
print(f"Valid JSON:           {n_json_valid}  ({n_json_valid / len(records) * 100:.2f}%)")
print(f"Invalid JSON:         {n_json_invalid}  ({n_json_invalid / len(records) * 100:.2f}%)")

GATE 1 — JSON PARSING
Total records:        3004
Valid JSON:           3004  (100.00%)
Invalid JSON:         0  (0.00%)


In [ ]:
BBox = conlist(float, min_length=4, max_length=4)

class RuleViolation(BaseModel):
    bounding_box: Optional[List[BBox]] = None
    reason: Optional[str] = None


class UnifiedOutput(BaseModel):
    """Mirrors data/schemas.py::UnifiedOutput — the model's required output contract."""
    caption: str
    rule_1_violation: Optional[RuleViolation] = None
    rule_2_violation: Optional[RuleViolation] = None
    rule_3_violation: Optional[RuleViolation] = None
    rule_4_violation: Optional[RuleViolation] = None
    excavator: List[BBox] = Field(default_factory=list)
    rebar: List[BBox] = Field(default_factory=list)
    worker_with_white_hard_hat: List[BBox] = Field(default_factory=list)


def validate_unified_output(parsed_data: Optional[Dict[str, Any]]):
    """Validates a parsed dict against the UnifiedOutput schema. Returns None on failure."""
    if parsed_data is None:
        return None
    try:
        return UnifiedOutput(**parsed_data)
    except Exception:
        return None


# Run Gate 2 over every record that passed Gate 1
schema_results = []
schema_errors = []  # keep the actual pydantic error message for diagnostics

for i, parsed in enumerate(parsed_results):
    if parsed is None:
        schema_results.append(None)
        schema_errors.append(None)
        continue
    try:
        validated = UnifiedOutput(**parsed)
        schema_results.append(validated)
        schema_errors.append(None)
    except Exception as e:
        schema_results.append(None)
        schema_errors.append(str(e))

n_schema_valid = sum(1 for s in schema_results if s is not None)
n_schema_invalid = len(records) - n_schema_valid

print("=" * 60)
print("GATE 2 — SCHEMA VALIDATION")
print("=" * 60)
print(f"Total records:            {len(records)}")
print(f"Valid schema:             {n_schema_valid}  ({n_schema_valid / len(records) * 100:.2f}%)")
print(f"Invalid schema:           {n_schema_invalid}  ({n_schema_invalid / len(records) * 100:.2f}%)")
print(f"  (of which failed Gate 1 already): {n_json_invalid}")
print(f"  (valid JSON but bad schema):      {n_schema_invalid - n_json_invalid}")

GATE 2 — SCHEMA VALIDATION
Total records:            3004
Valid schema:             3004  (100.00%)
Invalid schema:           0  (0.00%)
  (of which failed Gate 1 already): 0
  (valid JSON but bad schema):      0


In [ ]:
print("=" * 60)
print("COMBINED FUNNEL SUMMARY")
print("=" * 60)
print(f"{'Stage':<30} {'Count':>10} {'% of total':>12}")
print("-" * 54)
print(f"{'Total records':<30} {len(records):>10} {'100.00%':>12}")
print(f"{'Passed Gate 1 (JSON)':<30} {n_json_valid:>10} {n_json_valid/len(records)*100:>11.2f}%")
print(f"{'Passed Gate 2 (Schema)':<30} {n_schema_valid:>10} {n_schema_valid/len(records)*100:>11.2f}%")
print(f"{'Failed Gate 1 (JSON)':<30} {n_json_invalid:>10} {n_json_invalid/len(records)*100:>11.2f}%")
print(f"{'Failed Gate 2 only':<30} {n_schema_invalid - n_json_invalid:>10} {(n_schema_invalid - n_json_invalid)/len(records)*100:>11.2f}%")

if n_json_invalid == 0 and n_schema_invalid == 0:
    print("\nALL records passed both gates cleanly.")
else:
    print(f"\n{n_schema_invalid} / {len(records)} records will NOT contribute to strict-vs-valid")
    print("    metric differences downstream — inspect the failure breakdowns below.")

COMBINED FUNNEL SUMMARY
Stage                               Count   % of total
------------------------------------------------------
Total records                        3004      100.00%
Passed Gate 1 (JSON)                 3004      100.00%
Passed Gate 2 (Schema)               3004      100.00%
Failed Gate 1 (JSON)                    0        0.00%
Failed Gate 2 only                      0        0.00%

ALL records passed both gates cleanly.


In [ ]:
# Only look at records that had valid JSON but still failed schema
schema_only_failures = [
    (i, schema_errors[i]) for i in range(len(records))
    if parsed_results[i] is not None and schema_results[i] is None
]

print(f"Records with valid JSON but invalid schema: {len(schema_only_failures)}\n")

if schema_only_failures:
    # Extract the pydantic "field required" / "type" style short reason for grouping
    def short_reason(err_msg: str) -> str:
        lines = err_msg.strip().split("\n")
        # pydantic v2 errors look like: "<field>\n  <message> [type=..., ...]"
        reasons = [l.strip() for l in lines if "[type=" in l]
        return reasons[0].split("[type=")[0].strip() if reasons else lines[-1][:80]

    reason_counts = Counter(short_reason(err) for _, err in schema_only_failures)

    print("Most common schema-failure reasons:")
    for reason, count in reason_counts.most_common(10):
        print(f"  {count:>4}x  {reason}")
else:
    print("No schema-only failures — every valid-JSON record also matched the schema.")

Records with valid JSON but invalid schema: 0

No schema-only failures — every valid-JSON record also matched the schema.


In [ ]:
N_EXAMPLES = 50

print("=" * 60)
print(f"GATE 1 FAILURE EXAMPLES (up to {N_EXAMPLES})")
print("=" * 60)
gate1_failures = [i for i in range(len(records)) if parsed_results[i] is None]
for idx in gate1_failures[:N_EXAMPLES]:
    r = records[idx]
    print(f"\n--- image_id: {r.get('image_id', 'UNKNOWN')} (record #{idx}) ---")
    raw = r.get("raw_output", "")
    print(raw)

print("\n" + "=" * 60)
print(f"GATE 2 FAILURE EXAMPLES (up to {N_EXAMPLES})")
print("=" * 60)
gate2_only_failures = [i for i, _ in schema_only_failures]
for idx in gate2_only_failures[:N_EXAMPLES]:
    r = records[idx]
    print(f"\n--- image_id: {r.get('image_id', 'UNKNOWN')} (record #{idx}) ---")
    print("Parsed JSON:", json.dumps(parsed_results[idx], indent=2))
    print("Pydantic error:", schema_errors[idx])

GATE 1 FAILURE EXAMPLES (up to 50)

GATE 2 FAILURE EXAMPLES (up to 50)
